# 1. library

In [1]:
import numpy as np
import pandas as pd

# 2. import & process data

In [ ]:
# 1. import data
fitted_allspots_params = pd.read_csv("./processed_data/1_fitted_allspots_params.csv")
fitted_genealogy_data = pd.read_csv("./processed_data/2_fitted_genealogy_data_exponential.csv")

# 2. exp growth
def exp_growth(t, um2_biomass0, k):
    return um2_biomass0 * np.exp(k * t)

# 3. calculate FOV area
scale = 0.0645 # pixel to um
FOV_area_records = [
    dict(Specie="PY1", Condition="no-supernatant", replicate=1, FOV=1, FOV_pixel=1308*1014), 
    dict(Specie="PY1", Condition="no-supernatant", replicate=1, FOV=2, FOV_pixel=1311*1013),
    dict(Specie="PY1", Condition="no-supernatant", replicate=1, FOV=3, FOV_pixel=1309*1011),
    dict(Specie="PY1", Condition="no-supernatant", replicate=2, FOV=1, FOV_pixel=1253*999),
    dict(Specie="PY1", Condition="no-supernatant", replicate=2, FOV=2, FOV_pixel=1254*1011),
    dict(Specie="PY1", Condition="no-supernatant", replicate=2, FOV=3, FOV_pixel=1273*1016),
    dict(Specie="PY1", Condition="no-supernatant", replicate=3, FOV=1, FOV_pixel=1243*986),
    dict(Specie="PY1", Condition="no-supernatant", replicate=3, FOV=2, FOV_pixel=1238*1006),
    dict(Specie="PY1", Condition="no-supernatant", replicate=3, FOV=3, FOV_pixel=1239*976),
    dict(Specie="PY1", Condition="supernatant", replicate=1, FOV=1, FOV_pixel=1357*985),
    dict(Specie="PY1", Condition="supernatant", replicate=1, FOV=2, FOV_pixel=1343*995),
    dict(Specie="PY1", Condition="supernatant", replicate=1, FOV=3, FOV_pixel=1360*989),
    dict(Specie="PY1", Condition="supernatant", replicate=2, FOV=1, FOV_pixel=1290*956),
    dict(Specie="PY1", Condition="supernatant", replicate=2, FOV=2, FOV_pixel=1303*960),
    dict(Specie="PY1", Condition="supernatant", replicate=2, FOV=3, FOV_pixel=1295*957),
    dict(Specie="PY1", Condition="supernatant", replicate=3, FOV=1, FOV_pixel=1270*1000),
    dict(Specie="PY1", Condition="supernatant", replicate=3, FOV=2, FOV_pixel=1234*982),
    dict(Specie="PY1", Condition="supernatant", replicate=3, FOV=3, FOV_pixel=1260*1005),
    dict(Specie="europaea", Condition="no-supernatant", replicate=1, FOV=1, FOV_pixel=1369*1011),
    dict(Specie="europaea", Condition="no-supernatant", replicate=1, FOV=2, FOV_pixel=1368*1016),
    dict(Specie="europaea", Condition="no-supernatant", replicate=1, FOV=3, FOV_pixel=1368*1015),
    dict(Specie="europaea", Condition="no-supernatant", replicate=2, FOV=1, FOV_pixel=1360*988),
    dict(Specie="europaea", Condition="no-supernatant", replicate=2, FOV=2, FOV_pixel=1344*979),
    dict(Specie="europaea", Condition="no-supernatant", replicate=2, FOV=3, FOV_pixel=1360*981),
    dict(Specie="europaea", Condition="no-supernatant", replicate=3, FOV=1, FOV_pixel=1322*990),
    dict(Specie="europaea", Condition="no-supernatant", replicate=3, FOV=2, FOV_pixel=1331*1006),
    dict(Specie="europaea", Condition="no-supernatant", replicate=3, FOV=3, FOV_pixel=1328*1010)
]
FOV_area_df = pd.DataFrame(FOV_area_records)
FOV_area_df["FOV_area"] = FOV_area_df["FOV_pixel"] * (scale ** 2)  # convert to um^2

# 4. calculate average elongation rate
singleCell_elongation_rate = (
    fitted_genealogy_data
    .query("IF_will_Divide == 'divide' ")
    .groupby(['Specie', 'Condition', 'replicate', 'FOV', 'Time'])
    .agg(
        mean_elongation_rate = ('fitted_elongation_rate', 'mean'),
        std_elongation_rate = ('fitted_elongation_rate', 'std'),
        n_cells_used_for_elongation_rate = ('fitted_elongation_rate', 'count')
    )
    .reset_index()
)

# 5 calculate biomass
singleCell_elongation_rate_with_biomass_data = (
    singleCell_elongation_rate.
    merge(fitted_allspots_params[['Specie', 'Condition', 'replicate', 'FOV', 'um2_biomass0', 'k', 'cell_num_t0', 'max_time']],
          on=['Specie', 'Condition', 'replicate', 'FOV'],
          how='left'
          )
    )
# limit under max_time
singleCell_elongation_rate_with_biomass_data = (
    singleCell_elongation_rate_with_biomass_data
    .loc[
        singleCell_elongation_rate_with_biomass_data['Time']
        <= singleCell_elongation_rate_with_biomass_data['max_time']
    ]
)
singleCell_elongation_rate_with_biomass_data['um2_biomass'] = exp_growth(
    t=singleCell_elongation_rate_with_biomass_data['Time'],
    um2_biomass0=singleCell_elongation_rate_with_biomass_data['um2_biomass0'],
    k=singleCell_elongation_rate_with_biomass_data['k']
)

In [3]:
# 6. calculate ideal biomass production
biomass_data = (
    singleCell_elongation_rate_with_biomass_data
    .merge(FOV_area_df, how='left',
           on=['Specie', 'Condition', 'replicate', 'FOV'])
)
# Exclude data points where the elongation rate was calculated from fewer than 10 cells
biomass_data = biomass_data[biomass_data['n_cells_used_for_elongation_rate'] >= 10].copy()

# 6.1. calculate ideal biomass production
def compute_biomass_production(row):
    if row['Specie'] == "PY1":
        return row['um2_biomass'] * ( np.exp(row['mean_elongation_rate'] * 10/60*10) - 1 )
    elif row['Specie'] == "europaea":
        return row['um2_biomass'] * ( np.exp(row['mean_elongation_rate'] * 10/60*5) - 1 )
    else:
        return np.nan

biomass_data['um2_biomass_production'] = biomass_data.apply(compute_biomass_production, axis=1)
biomass_data['sum_um2_biomass_production'] = (
    biomass_data
    .groupby(['Specie', 'Condition', 'replicate', 'FOV'])['um2_biomass_production']
    .cumsum()
)

# calculated from previous study (Delince et al., lab chip, 2016) using klyaout
device_area = 1.47*10**8 
def compute_density(row, target_col):
    if row['Specie'] == "PY1":
        return (
        ( (row[target_col] * 750 * 1e-3) * device_area / row['FOV_area'] ) # convert to um^3, 750 is the conversion factor from um^2 to um^3 for a 750 nm thick layer.
        / ( 8 * 1e-3 * 100 ) ) # convert to mL, 1e-3 is the conversion factor from µL to mL. 8 µL/min, analyze per 100 min.
    elif row['Specie'] == "europaea":
        return ( 
        ( (row[target_col] * 750 * 1e-3) * device_area / row['FOV_area'] ) # convert to um^3, 750 is the conversion factor from um^2 to um^3 for a 750 nm thick layer.
        / ( 8 * 1e-3 * 50 ) # convert to mL, 1e-3 is the conversion factor from µL to mL. 8 µL/min, analyze per 50 min.
    )
    else:
        return None

# 6.2. calculate ideal biomass production volume density
biomass_data['um3_biomass_production_density'] = biomass_data.apply(
    lambda row: compute_density(row, 'um2_biomass_production'), axis=1
)

# 6.3. calculate initial cell density
biomass_data['initial_cell_density'] = biomass_data.apply(
    lambda row: compute_density(row, 'cell_num_t0'), axis=1
)

In [4]:
# 7. merge genealogy and biomass data
merge_data = (
    fitted_genealogy_data
    .merge(biomass_data, how='inner',
           on=['Specie', 'Condition', 'replicate', 'FOV', 'Time'])
)

In [5]:
# 8. daughter cell pair infomation
daughter_cell_pair = (
    fitted_genealogy_data
    .query("IF_will_Divide == 'divide' ")
    .groupby(['Specie', 'Condition', 'replicate', 'FOV', 'generation_count', 'Source.spot.ID'])
    .filter(lambda x: len(x) == 2)
    .reset_index(drop=True)
)

daughter_cell_pair['div_ratio'] = (
    daughter_cell_pair
    .groupby(['Specie', 'Condition', 'replicate', 'FOV', 'generation_count', 'Source.spot.ID'])['Area']
    .transform(lambda x: x / x.sum())
)

In [6]:
# 9. summarize data
def summarize_data(merge_data):
    summary_data = (
        merge_data
        .sort_values(['Specie', 'Condition', 'replicate', 'FOV', 'track', 'generation_count', 'Frame'])
        .groupby(['Specie', 'Condition', 'replicate', 'FOV', 'track', 'generation_count'])
        .agg(
            first_spot_ID = ('Source.spot.ID', 'first'),
            Last_spot_ID = ('Source.spot.ID', 'last'),
            Source_spot_ID = ('Source.spot.ID', 'first'),
            IF_Divided = ('IF_Divided', 'first'),
            IF_will_Divide = ('IF_will_Divide', 'first'),
            
            generation_time = ('generation_time', 'first'),
            fitted_elongation_rate = ('fitted_elongation_rate', 'first'),
            fitted_Ad = ('fitted_area', 'last'),
            fitted_Ab = ('fitted_area', 'first'),
            Td = ('Time_end', 'first'),
            Tb = ('Time_start', 'first'),
            
            um3_biomass_production_density_start = ('um3_biomass_production_density', 'first'),
        )
        .reset_index()
    )

    # 10.1. calculate additional information
    summary_data['fitted_deltaA'] = (
        summary_data['fitted_Ad'] - summary_data['fitted_Ab']
    )

    # 10.2. filter summary_data, only fitted_elongation_rate > 0 is seleceted for latter analysis.
    filtered_summary_data = summary_data[summary_data['fitted_elongation_rate'] > 0]

    return filtered_summary_data

In [7]:
summary_data = summarize_data(merge_data)

biomass_data.to_csv("./processed_data/3_biomass_data.csv", index=False)
merge_data.to_csv("./processed_data/4_fitted_genealogy_data_exponential_mutate.csv", index=False)
summary_data.to_csv("./processed_data/5_summary_data_exponential.csv", index=False)
daughter_cell_pair.to_csv("./processed_data/6_daughter_cell_pair.csv", index=False)